In [1]:
import sys
sys.path.append("..")
import torch
import json
import numpy as np
from pathlib import Path
from tqdm import tqdm
import gc
from datasets import load_from_disk

from src.data import PROJECT_ROOT
from src.llm_upgrade import wrap_for_transformer_lens
from src.dataset_cot import prepare_direct_pythia_dataset
from src.sae_test import TopKSAE, collect_all_activations, create_activation_dataloader, save_activations, load_activations

# Параметры построения SAE

In [2]:
EXP_ID = "exp8-2"
MODEL_NAME = "EleutherAI/pythia-1b-deduped"
VARIANT = "depth-1"
ADAPTER_PATH = str(PROJECT_ROOT / f"results/checkpoints/finetune/{EXP_ID}/checkpoint-1000")

In [3]:
PROBING_JSON = PROJECT_ROOT / "results/probing/probe_1b(seq_qlora)_depth-1_resid_post_last.json"
with open(PROBING_JSON, "r", encoding="utf-8") as f:
    probing_data = json.load(f)
BEST_LAYER = probing_data["summary"]["best_layer"]
print(f"Загружен слой: {BEST_LAYER} (probing acc: {probing_data['summary']['best_dev_acc']:.4f})")

Загружен слой: 13 (probing acc: 0.6040)


In [4]:
HOOK_POINT = f"blocks.{BEST_LAYER}.hook_resid_post"

In [5]:
D_SAE = 16384       #
K = 128             # жёсткая разреженность (Top-K)
LR = 3e-4           # скорость обучения
BATCH_SIZE = 128    # размер батча активаций
MAX_LENGTH = 512

In [6]:
SAVE_DIR = PROJECT_ROOT / f"results/sae/{EXP_ID}_centered/layer_{BEST_LAYER}"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

In [7]:
torch.cuda.empty_cache()
gc.collect()

80

# Загрузка модели и датасета

In [8]:
# Загрузка модели и данных
hooked_model, tokenizer = wrap_for_transformer_lens(MODEL_NAME, ADAPTER_PATH, device="cpu")
hooked_model.eval()
hooked_model.to("cuda")

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model EleutherAI/pythia-1b-deduped into HookedTransformer
Moving model to device:  cuda


HookedTransformer(
  (embed): Embed()
  (hook_embed): HookPoint()
  (blocks): ModuleList(
    (0-15): 16 x TransformerBlock(
      (ln1): LayerNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln2): LayerNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (attn): Attention(
        (hook_k): HookPoint()
        (hook_q): HookPoint()
        (hook_v): HookPoint()
        (hook_z): HookPoint()
        (hook_attn_scores): HookPoint()
        (hook_pattern): HookPoint()
        (hook_result): HookPoint()
        (hook_rot_k): HookPoint()
        (hook_rot_q): HookPoint()
      )
      (mlp): MLP(
        (hook_pre): HookPoint()
        (hook_post): HookPoint()
      )
      (hook_attn_in): HookPoint()
      (hook_q_input): HookPoint()
      (hook_k_input): HookPoint()
      (hook_v_input): HookPoint()
      (hook_mlp_in): HookPoint()
      (hook_attn_out): HookPoint()
      (hook_mlp_out): HookPoint()
    

In [9]:
cache_path = PROJECT_ROOT / "data" / "processed" / f"ruletaker_{VARIANT}"
raw_dataset = load_from_disk(str(cache_path))
train_dataset = prepare_direct_pythia_dataset(raw_dataset["train"])

In [10]:
d_model = hooked_model.cfg.d_model

In [11]:
print(f"Датасет: {len(train_dataset)} примеров | d_model={d_model}")

Датасет: 61699 примеров | d_model=2048


# Сбор активаций

In [12]:
# Путь для сохранения активаций
ACTIVATIONS_PATH = PROJECT_ROOT / f"results/sae/{EXP_ID}/layer_{BEST_LAYER}" / "activations.pt"

In [13]:
# all_activations = collect_all_activations(
#     dataset=train_dataset,
#     model=hooked_model,
#     tokenizer=tokenizer,
#     hook_point=HOOK_POINT,
#     max_length=MAX_LENGTH,
#     collect_batch_size=4,  # Маленький батч для сбора
#     prepend_bos=True,
#     device="cuda"
# )

In [14]:
# metadata = {
#     "model": MODEL_NAME,
#     "variant": VARIANT,
#     "layer": BEST_LAYER,
#     "hook_point": HOOK_POINT,
#     "max_length": MAX_LENGTH,
#     "n_samples": len(all_activations)
# }
# save_activations(all_activations, ACTIVATIONS_PATH, metadata=metadata)

In [15]:
activations_data = torch.load(ACTIVATIONS_PATH, map_location="cpu", weights_only=False)
all_activations = activations_data["activations"].float()

In [16]:
# Сбор статистик для диагностики
mean = all_activations.mean(dim=0, keepdim=True)
scale = (all_activations - mean).norm(dim=1).mean()
print(f"Статистики: mean.shape={mean.shape}, scale={scale:.4f}")

Статистики: mean.shape=torch.Size([1, 2048]), scale=34.6743


In [17]:
# Сбор статистик для диагностики
act_mean = all_activations.mean(dim=0)
acts_centered = all_activations - act_mean
torch.save(act_mean, SAVE_DIR / "act_mean.pt")
print(f"Среднее активации: ||act_mean||={act_mean.norm().item():.4f}")

Среднее активации: ||act_mean||=77.0557


In [18]:
# Создание DataLoader для обучения
train_loader = create_activation_dataloader(all_activations, batch_size=BATCH_SIZE, shuffle=True)
print(f"{len(train_loader)} батчей")

483 батчей


# SAE

In [19]:
# Очистка памяти перед запуском
torch.cuda.empty_cache()
gc.collect()

78

In [20]:
print(f"Инициализация SAE: d_in={d_model}, d_sae={D_SAE}, k={K}, mean={act_mean}, batch_size={BATCH_SIZE}")

Инициализация SAE: d_in=2048, d_sae=16384, k=128, mean=tensor([ 1.5081,  0.3763,  0.5373,  ..., -1.2110, -1.3594, -0.3295]), batch_size=128


In [21]:
# Инициализация модели и оптимизатора
sae = TopKSAE(d_in=d_model, d_sae=D_SAE, k=K, mean=act_mean).to("cuda")
optimizer = torch.optim.Adam(sae.parameters(), lr=LR, betas=(0.9, 0.999))

In [22]:
# Параметры обучения
NUM_EPOCHS = 10
total_steps = len(train_loader) * NUM_EPOCHS
log_interval, save_interval = 200, 500
loss_history = []
global_step = 0

In [23]:
print(f"Всего шагов: {total_steps}")

Всего шагов: 4830


In [24]:
for epoch in range(NUM_EPOCHS):
    for batch_acts, in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}"):
        batch_acts = batch_acts.to("cuda")  # Быстрая передача благодаря pin_memory

        optimizer.zero_grad()
        sparse, recon = sae(batch_acts)
        loss = (recon - batch_acts).pow(2).sum(dim=-1).mean()
        loss.backward()

        # Нормализация декодера после каждого шага
        with torch.no_grad():
            sae.W_dec.data /= sae.W_dec.data.norm(dim=0, keepdim=True)
        optimizer.step()

        loss_history.append(loss.item())
        if global_step % log_interval == 0:
            print(f"Step {global_step:5d} | Loss: {loss.item():.4f}")
        if global_step % save_interval == 0 and global_step > 0:
            torch.save({
                "step": global_step,
                "model_state_dict": sae.state_dict(),
                "loss_history": loss_history
            }, f"{SAVE_DIR}/checkpoint_{global_step}.pt")
        global_step += 1

Epoch 1/10:   1%|          | 5/483 [00:00<01:04,  7.42it/s]

Step     0 | Loss: 1285.5181


Epoch 1/10:  43%|████▎     | 206/483 [00:05<00:06, 40.35it/s]

Step   200 | Loss: 114.7361


Epoch 1/10:  84%|████████▍ | 408/483 [00:10<00:01, 39.85it/s]

...

Epoch 10/10:  54%|█████▍    | 260/483 [00:08<00:06, 32.88it/s]

Step  4600 | Loss: 35.9232


Epoch 10/10:  95%|█████████▌| 460/483 [00:14<00:00, 29.35it/s]

Step  4800 | Loss: 36.4628


Epoch 10/10: 100%|██████████| 483/483 [00:15<00:00, 30.64it/s]


In [25]:
# Сохранение финального шага
torch.save({
    "step": global_step,
    "model_state_dict": sae.state_dict(),
    "loss_history": loss_history,
    "config": {
        "d_in": d_model,
        "d_sae": D_SAE,
        "k": K,
        "epochs": NUM_EPOCHS,
        "batch_size": BATCH_SIZE
    }
}, f"{SAVE_DIR}/sae_final.pt")

# Диагностика качества SAE

In [26]:
# Загрузка финальной модели
checkpoint = torch.load(f"{SAVE_DIR}/sae_final.pt", map_location="cuda", weights_only=False)
sae.load_state_dict(checkpoint["model_state_dict"])
sae.eval()

TopKSAE()

In [27]:
# Тестовый батч для оценки
test_texts = [train_dataset[i]["text"] for i in range(128)]
tokens = hooked_model.to_tokens(test_texts, prepend_bos=True).to("cuda")
if MAX_LENGTH and tokens.shape[1] > MAX_LENGTH:
    tokens = tokens[:, :MAX_LENGTH]

In [28]:
with torch.no_grad():
    _, cache = hooked_model.run_with_cache(tokens, names_filter=lambda n: n == HOOK_POINT)
    acts = cache[HOOK_POINT]
    lengths = [len(tokenizer.encode(t, add_special_tokens=False)) for t in test_texts]
    batch_acts = torch.stack([acts[i, min(lengths[i], acts.shape[1]-1), :] for i in range(len(test_texts))]).to(torch.float32)

In [29]:
# Прогон через SAE
sparse, recon = sae(batch_acts)

In [30]:
# Метрики
var_explained = 1.0 - ((batch_acts - recon).pow(2).sum() / (batch_acts - batch_acts.mean(dim=0)).pow(2).sum()).item()
sparsity = (sparse != 0).float().mean().item()
print(f"Explained Variance (centered): {var_explained:.4f}")
print(f"Sparsity (L0): {sparsity:.4f} | Target: {K/D_SAE:.4f}")

Explained Variance (centered): 0.9690
Sparsity (L0): 0.0078 | Target: 0.0078


In [31]:
# # Диагностика: проверка forward и метрик
print("=== Проверка SAE.forward ===")
test_batch = all_activations[:32].to("cuda")  # сырые активации, [32, d_model]
print(f"Input norm (raw): {test_batch.norm():.4f}")

sparse, recon = sae(test_batch)
print(f"Recon norm (raw): {recon.norm():.4f}")
print(f"Residual norm: {(recon - test_batch).norm():.4f}")
print(f"Mean buffer: {sae.mean.norm() if sae.mean is not None else 'None'}")

# Корректная формула EV для любых данных (сырых или центрированных)
def compute_ev(data, recon):
    # EV = 1 - MSE / Var(data), где Var считается вокруг среднего data
    mse = ((data - recon) ** 2).sum()
    var = ((data - data.mean(dim=0)) ** 2).sum()
    return 1 - mse / var

ev_correct = compute_ev(test_batch, recon).item()
print(f"Explained Variance (корректно): {ev_correct:.4f}")

# Сравнение с исходной формулой (может быть некорректной)
ev_original = 1 - ((test_batch - recon).pow(2).sum() / (test_batch - test_batch.mean(dim=0)).pow(2).sum()).item()
print(f"Explained Variance (исходная формула): {ev_original:.4f}")

=== Проверка SAE.forward ===
Input norm (raw): 478.3105
Recon norm (raw): 477.5195
Residual norm: 37.2857
Mean buffer: 77.05574035644531
Explained Variance (корректно): 0.9692
Explained Variance (исходная формула): 0.9692
